# Fine-tune usas-slm (Qwen3-4B-Instruct-2507 + LoRA, Unsloth)
Học model nhỏ **tách tin nhắn thành các ý** (xem `advisor/app/skills/slm_intents.md`).

**Cần có trên máy bạn**: `train.jsonl` và `val.jsonl` (ở `advisor/data/slm/finetune/`). Notebook sẽ hiện hộp thoại để bạn chọn 2 file này tải lên Colab.
**Ra**: file `.gguf` (model đã nén) + `training_log.csv` + `trainer_state.json` + bản LoRA, lưu vào Google Drive `MyDrive/usas_slm/<VERSION>/` (Drive chỉ dùng để LƯU kết quả).

Làm lần lượt từng ô, từ trên xuống. Đọc `HUONG_DAN_FINETUNE.md` để hiểu từng bước.

| Bước | Việc | Thời gian |
|---|---|---|
| 1-2 | Bật GPU, cài thư viện | vài phút |
| 3-4 | Tải dữ liệu, đặt thông số | vài phút |
| 5-10 | Tải model, gắn LoRA, chuẩn bị dữ liệu, **kiểm tra trước khi train** | 5-10 phút |
| 11 | **Huấn luyện** (bước duy nhất lâu) | vài giờ, tùy GPU |
| 12-14 | Lưu nhật ký, thử nhanh, xuất file `.gguf` | 10-20 phút |

**Bị ngắt kết nối giữa chừng?** Không mất gì: checkpoint nằm trên Drive. Kết nối lại, chạy lại **Bước 1 đến 10** (kể cả chọn lại 2 file ở Bước 3, **giữ nguyên `VERSION`**), rồi chạy Bước 11, nó tự tiếp tục từ checkpoint mới nhất.

## Bước 1. Bật GPU
Menu **Runtime → Change runtime type → chọn GPU (T4 hoặc tốt hơn) → Save**. Ô dưới phải in ra tên GPU, nếu báo lỗi là chưa bật GPU.

In [ ]:
!nvidia-smi

## Bước 2. Cài thư viện
Unsloth giúp train nhanh và ít bộ nhớ. Mất vài phút. Nếu Colab bảo khởi động lại (Restart session) thì khởi động lại rồi chạy tiếp từ Bước 3.

In [ ]:
%%capture
!pip install unsloth

## Bước 3. Tải 2 file dữ liệu từ máy bạn lên Colab, và nối Drive để lưu kết quả
Chạy ô dưới, khi hiện nút **Choose files** thì chọn **cả hai** file `train.jsonl` và `val.jsonl` (giữ phím Ctrl để chọn hai file cùng lúc). Chờ tải xong (mỗi file vài chục MB).
Drive được nối chỉ để lưu kết quả train, nên không cần tạo thư mục gì trước, ô tự tạo.

In [ ]:
from google.colab import drive, files
import os, shutil

VERSION = "v5"   # tên lần train. Mỗi lần train bằng DỮ LIỆU MỚI phải đổi tên (v6, v7...) để không tiếp tục nhầm từ checkpoint của lần trước.
                 # Bị ngắt kết nối và muốn chạy tiếp cùng một lần train thì GIỮ NGUYÊN tên này.
drive.mount("/content/drive")
OUT_DIR = f"/content/drive/MyDrive/usas_slm/{VERSION}"  # nơi lưu kết quả (ô tự tạo)
os.makedirs(OUT_DIR, exist_ok=True)

DATA_DIR = "/content/data"                              # nơi để 2 file vừa tải lên
os.makedirs(DATA_DIR, exist_ok=True)

print("Chọn 2 file train.jsonl và val.jsonl từ máy bạn...")
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f"{DATA_DIR}/{name}")

print(os.listdir(DATA_DIR))  # phải thấy train.jsonl và val.jsonl
missing = [n for n in ("train.jsonl", "val.jsonl") if not os.path.exists(f"{DATA_DIR}/{n}")]
assert not missing, f"Thiếu file: {missing}. Chạy lại ô này và chọn đủ 2 file."

# Kiểm nhanh đúng phiên bản dữ liệu: bản mới có khóa "intents" trong đáp án, bản cũ của model v3 dùng "cac_y".
# Phải đọc JSON thật: trong dòng thô, đáp án nằm trong một chuỗi nên dấu ngoặc kép bị thêm gạch chéo
import json
row = json.loads(open(f"{DATA_DIR}/train.jsonl", encoding="utf-8").readline())
answer = json.loads(row["messages"][2]["content"])
assert "intents" in answer, "Đây có vẻ là file train CŨ (khóa tiếng Việt). Hãy tải bản mới ở advisor/data/slm/finetune/."
print("OK: dữ liệu đúng phiên bản mới")

## Bước 4. Các thông số (chỉnh ở đây, các ô sau tự dùng)

In [ ]:
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"  # model gốc (bản Instruct, không có chế độ suy luận)
MAX_SEQ = 4096        # độ dài tối đa của một hội thoại (token)
EPOCHS = 2            # số lượt đi qua toàn bộ dữ liệu train (tối đa, có dừng sớm). 2 là hợp lý: dữ liệu viết theo khuôn nên học nhanh, thêm lượt dễ học thuộc và rất tốn giờ GPU
LR = 2e-4             # tốc độ học
LORA_R = 16           # "độ rộng" của phần được học thêm (LoRA)
BATCH = 2             # số hội thoại xử lý cùng lúc
GRAD_ACCUM = 8        # gom 8 lần rồi mới cập nhật: batch hiệu dụng = 2 x 8 = 16
EVAL_EVERY = 100      # cứ 100 bước kiểm val loss và lưu checkpoint một lần
VAL_MAX = 300         # chỉ dùng 300 mẫu val ngẫu nhiên để mỗi lần kiểm nhanh (val đầy đủ dùng khi đo)
PATIENCE = 4          # val loss không tốt hơn sau 4 lần kiểm liên tiếp thì dừng
SEED = 3407
RUN_NAME = f"usas-slm-{VERSION}"   # tên model khi xuất .gguf và nạp vào Ollama

## Bước 5. Tải model gốc (nén 4-bit để vừa GPU)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,   # nén trọng số xuống 4 bit để tiết kiệm bộ nhớ
    dtype=None,          # để thư viện tự chọn kiểu số phù hợp GPU
)

## Bước 6. Gắn LoRA
Không sửa toàn bộ 4 tỷ tham số của model, chỉ thêm các "miếng vá" nhỏ (LoRA) rồi học các miếng này. Nhanh, nhẹ, ít bị quên kiến thức cũ.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_R,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # đổi thêm thời gian lấy bộ nhớ
    random_state=SEED,
)

## Bước 7. Đọc dữ liệu và ghép thành chữ theo khuôn của model
Mỗi dòng có 3 tin: system (prompt), user (tin học sinh), assistant (đáp án đúng). Hàm `apply_chat_template` ghép thành một chuỗi theo đúng khuôn mà Qwen quen dùng.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

data = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl", "val": f"{DATA_DIR}/val.jsonl"})


def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}


data = data.map(to_text, batched=True)
data["val"] = data["val"].shuffle(seed=SEED).select(range(min(VAL_MAX, len(data["val"]))))  # rút gọn tập val để kiểm nhanh
print(data)
print(data["train"][0]["text"][:1500])

## Bước 8. Kiểm tra độ dài (không được vượt MAX_SEQ)
Hội thoại dài hơn `MAX_SEQ` sẽ bị cắt mất đáp án, model học sai. Kết quả in ra phải có `lớn nhất` nhỏ hơn 4096.

In [ ]:
lengths = [len(tokenizer(t)["input_ids"]) for t in data["train"]["text"]]   # kiểm TOÀN BỘ tập train, từng mẫu một (mất khoảng 1 phút)
print("số token: trung bình", sum(lengths) // len(lengths), "| lớn nhất", max(lengths), "| giới hạn", MAX_SEQ)
assert max(lengths) < MAX_SEQ, "Có hội thoại dài quá MAX_SEQ: đáp án sẽ bị cắt. Tăng MAX_SEQ hoặc rút ngắn dữ liệu."
print("OK: không hội thoại nào bị cắt")

## Bước 9. Cấu hình huấn luyện
- **train_on_responses_only**: chỉ tính lỗi trên phần đáp án (assistant), không bắt model học thuộc prompt và tin học sinh.
- **Dừng sớm**: nếu val loss không còn tốt hơn thì dừng, và giữ lại điểm tốt nhất.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data["train"],
    eval_dataset=data["val"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        warmup_ratio=0.05,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        eval_strategy="steps",
        eval_steps=EVAL_EVERY,
        save_strategy="steps",
        save_steps=EVAL_EVERY,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=10,
        seed=SEED,
        report_to="none",
        output_dir=f"{OUT_DIR}/checkpoints",  # lưu checkpoint thẳng lên Drive để mất kết nối vẫn chạy tiếp được
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

## Bước 10. Kiểm tra phần bị che
Ô này in ra một mẫu với phần **không tính lỗi** để trống. Phải thấy chỉ còn đáp án JSON `{"intents": ...}`, không còn prompt hay tin học sinh. Nếu thấy cả prompt thì cấu hình `instruction_part` / `response_part` ở Bước 9 sai.

In [ ]:
sample = trainer.train_dataset[5]["labels"]
shown = tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in sample]).replace(tokenizer.pad_token, " ")
print(shown)
assert "PHÂN LOẠI TIN NHẮN" not in shown and '"intents"' in shown, "Che sai: còn prompt hoặc mất đáp án. Kiểm tra instruction_part / response_part ở Bước 9."
print("OK: chỉ còn đáp án JSON được tính lỗi")

## Bước 11. Chạy huấn luyện
Đây là bước lâu nhất. Số bước = (số mẫu train / 16) x EPOCHS (khoảng 267 bước mỗi epoch). Trên Colab T4 mỗi bước khoảng 40 giây (gồm cả lúc kiểm val), nên 2 epoch mất vài giờ; GPU mạnh hơn (L4, A100 của Colab Pro) nhanh hơn nhiều.

Cứ 10 bước in một dòng `loss` (lỗi trên train), cứ 100 bước in `eval_loss` (lỗi trên val) và lưu checkpoint lên Drive. `eval_loss` giảm rồi đi ngang là bình thường. Nếu `eval_loss` tăng lại trong khi `loss` vẫn giảm là model bắt đầu học thuộc, dừng sớm sẽ tự cắt và giữ điểm tốt nhất.

**Giữ tab mở và máy không ngủ.** Nếu bị ngắt, xem hướng dẫn ở đầu notebook.

In [ ]:
import glob
has_checkpoint = bool(glob.glob(f"{OUT_DIR}/checkpoints/checkpoint-*"))
print("Có checkpoint cũ trên Drive, sẽ TIẾP TỤC từ đó" if has_checkpoint else "Chưa có checkpoint, train từ đầu")
stats = trainer.train(resume_from_checkpoint=has_checkpoint)  # có checkpoint cũ trên Drive thì chạy tiếp, không làm lại từ đầu
print(stats.metrics)

## Bước 12. Lưu nhật ký train (để vẽ biểu đồ cho thầy)

In [ ]:
import json, csv, shutil

log = trainer.state.log_history
keys = sorted({k for row in log for k in row})
with open(f"{OUT_DIR}/training_log.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    w.writerows(log)
with open(f"{OUT_DIR}/trainer_state.json", "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=1)
print("đã lưu nhật ký,", len(log), "dòng; val loss tốt nhất:", trainer.state.best_metric)

# Lưu thêm bản LoRA (nhỏ, vài chục MB) để còn dùng lại nếu bước xuất .gguf ở dưới bị lỗi
model.save_pretrained(f"{OUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUT_DIR}/lora")
print("đã lưu bản LoRA vào", f"{OUT_DIR}/lora")

## Bước 13. Thử nhanh model vừa train
Dùng đúng prompt trong dữ liệu train. Kiểm tra bằng mắt: có ra JSON đúng khuôn, tách đủ ý không. Đây chỉ là thử nhanh, **không phải phép đo** (phép đo chính thức chạy bằng script trên tập test).

In [ ]:
FastLanguageModel.for_inference(model)
system_prompt = data["train"][0]["messages"][0]["content"]


def ask(question, profile=None, pending="", context=None):
    user = {"question": question, "profile": profile, "pending_question": pending}
    if context:
        user["context"] = context
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(user, ensure_ascii=False, separators=(",", ":"))},
    ]
    ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=400, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


for q in [
    "đánh giá hồ sơ của em và I-20 là gì vậy ạ",
    "so sánh 2 trường Purdue và Duke",
    "Purdue có ký túc xá cho du học sinh không",
    "em GPA 8.5, SAT 1420, ai sửa bài luận giúp em, OPT là gì",
]:
    print(q)
    print("  ->", ask(q))

## Bước 14. Xuất model nén (.gguf) để chạy bằng Ollama
`q4_k_m` nén còn khoảng 2,5 GB. Ô này mất vài phút và dùng nhiều ổ đĩa của Colab.

In [ ]:
model.save_pretrained_gguf(RUN_NAME, tokenizer, quantization_method="q4_k_m")

import glob
found = glob.glob("/content/**/*.gguf", recursive=True) + glob.glob("/content/**/Modelfile", recursive=True)
print(found)
for path in found:
    shutil.copy(path, OUT_DIR)
print("đã chép vào", OUT_DIR, os.listdir(OUT_DIR))

## Xong
Tải file `.gguf` và `Modelfile` từ `MyDrive/usas_slm/<VERSION>/` về máy (hoặc đẩy lên Hugging Face), rồi làm theo mục "Sau khi train" trong `HUONG_DAN_FINETUNE.md`.